<a href="https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-02-grafos/12_Matrizes_de_Incidencia_Adjacencia_e_Custos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 12 - Notebook: Matrizes de Incidência, Adjacência e Custos na Rede Pneumática da Envasadora

**UNIFEI — Engenharia de Controle e Automação | Disciplina: Automática**
**Projeto: SCADA - Máquina de Envasamento de Copos Plásticos**

Neste notebook a máquina de envase é modelada como **dois grafos dirigidos**:

1. **Rede pneumática de distribuição de ar comprimido** (`FRL-001` → ilhas de válvulas → cilindros A–H, bico e ejetor de vácuo). Sobre ela construímos a **Matriz de Incidência** $B \in \{-1,0,1\}^{n \times m}$, a **Matriz de Adjacência** $A$, a **Matriz de Custos** $W$ e resolvemos o **balanço de massa de ar** $B \cdot \vec{Q} = \vec{S}$.
2. **Grafo de estações da mesa indexadora** (Dispensa → Envase → Tampa → Selagem → Ejeção), usado para analisar o fluxo de copos, potências da matriz de adjacência e o gargalo de produção.

### Melhorias em relação ao notebook original (planta NPK)
| # | Melhoria | Onde |
|---|---|---|
| M1 | Vazões **calculadas fisicamente** a partir de diâmetro, curso, pressão e cadência de cada cilindro (antes eram valores arbitrários) | Seção 2 |
| M2 | Impressão das matrizes de **adjacência binária e de custos** (antes eram construídas, mas nunca exibidas) | Seção 4 |
| M3 | Cálculo automático do **posto de $B$** e da **dimensão do espaço de ciclos** $m-n+1$ | Seção 3 |
| M4 | **Floyd-Warshall** com reconstrução de rotas (antes apenas citado na teoria) | Seção 4 |
| M5 | Solução do sistema **subdeterminado** com a **2ª Lei de Kirchhoff** (equação de malha) — o vetor $\vec{Q}$ deixa de ser "chutado" | Seção 5 |
| M6 | **Detecção de vazamento** por resíduo $\vec{r} = B\vec{Q}_{med} - \vec{S}$ (responde à Atividade 1) | Seção 6 |
| M7 | Verificação numérica de $L = BB^T = D - A$ (responde à Atividade 2) | Seção 7 |
| M8 | Grafo da mesa: potências $A^k$, lead time e **gargalo / cadência máxima** | Seção 8 |

## 1. Utilitários (tabelas ASCII e álgebra linear em Python puro)

Mantemos a filosofia do notebook original: **sem NumPy**, para que cada operação matricial fique explícita.

In [9]:
from dataclasses import dataclass
from fractions import Fraction
from typing import Any, Dict, List
import math

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols, casas=None) -> str:
    """Formata matriz 2D em tabela ASCII pura (suporta inf e casas decimais)."""
    def fmt(v):
        if v == float("inf"):
            return "∞"
        if isinstance(v, float):
            return f"{v:.{casas}f}" if casas is not None else (str(int(v)) if v.is_integer() else f"{v:g}")
        return str(v)
    txt = [[fmt(matriz[i][j]) for j in range(len(rotulos_cols))] for i in range(len(rotulos_linhas))]
    larguras = [max([len(str(c)), 3] + [len(txt[i][j]) for i in range(len(rotulos_linhas))])
                for j, c in enumerate(rotulos_cols)]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(f"{txt[i][j]:>{larguras[j]}}" for j in range(len(rotulos_cols))))
    return "\n".join(linhas)

def transposta(M):
    return [list(col) for col in zip(*M)]

def mult_mat(X, Y):
    Yt = transposta(Y)
    return [[sum(a * b for a, b in zip(lin, col)) for col in Yt] for lin in X]

def mult_mat_vet(M, v):
    return [sum(a * b for a, b in zip(lin, v)) for lin in M]

def posto_matriz(M) -> int:
    """Posto exato por eliminação de Gauss-Jordan com frações (sem erro de arredondamento)."""
    A = [[Fraction(x) for x in lin] for lin in M]
    n_lin, n_col = len(A), len(A[0])
    lin = 0
    for col in range(n_col):
        piv = next((r for r in range(lin, n_lin) if A[r][col] != 0), None)
        if piv is None:
            continue
        A[lin], A[piv] = A[piv], A[lin]
        for r in range(n_lin):
            if r != lin and A[r][col] != 0:
                f = A[r][col] / A[lin][col]
                A[r] = [a - f * b for a, b in zip(A[r], A[lin])]
        lin += 1
        if lin == n_lin:
            break
    return lin

def resolver_sistema_linear(M, b):
    """Eliminação de Gauss com pivoteamento parcial (sistema quadrado)."""
    n = len(M)
    A = [[float(x) for x in M[i]] + [float(b[i])] for i in range(n)]
    for col in range(n):
        piv = max(range(col, n), key=lambda r: abs(A[r][col]))
        if abs(A[piv][col]) < 1e-12:
            raise ValueError("Sistema singular: faltam equações independentes")
        A[col], A[piv] = A[piv], A[col]
        for r in range(col + 1, n):
            f = A[r][col] / A[col][col]
            for c in range(col, n + 1):
                A[r][c] -= f * A[col][c]
    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = (A[i][n] - sum(A[i][c] * x[c] for c in range(i + 1, n))) / A[i][i]
    return x

print("Utilitários carregados.")

Utilitários carregados.


## 2. Modelagem da Rede Pneumática como Dígrafo $G=(V,E)$  *(Melhoria M1)*

* **Vértices $V$:** a unidade de tratamento de ar `FRL-001` (fonte), as duas ilhas de válvulas (`ILHA-A` → Setores 100/200, `ILHA-B` → Setores 300/400/500) e os 10 consumidores pneumáticos.
* **Arestas $E$:** mangueiras de poliuretano, cada uma associada à válvula solenoide (tag ISA-5.1) que a comanda.
* **Linha de equalização** `ILHA-A → ILHA-B`: fecha um **anel** na distribuição, boa prática para reduzir queda de pressão quando a prensa e o elevador atuam simultaneamente.

O consumo de ar livre de um cilindro de dupla ação (avanço + recuo, desprezando a haste) é:

$$Q_{cil} = 2 \cdot \frac{\pi d^2}{4} \cdot s \cdot \frac{p_{man} + p_{atm}}{p_{atm}} \cdot f_{ciclos} \quad [\text{NL/min}]$$

In [10]:
P_ATM = 1.013            # bar (abs)
P_REDE = 6.0             # bar (manométrico) regulado no FRL-001
CADENCIA = 12.0          # copos/min (cadência nominal de projeto)
RAZAO_COMPRESSAO = (P_REDE + P_ATM) / P_ATM
MU_AR = 1.81e-5          # Pa.s (viscosidade dinâmica do ar a 20 °C)

@dataclass
class ConsumidorPneumatico:
    tag: str
    descricao: str
    setor: str
    diametro_mm: float = 0.0
    curso_mm: float = 0.0
    consumo_continuo_nl_min: float = 0.0   # p/ ejetor de vácuo
    fator_utilizacao: float = 1.0          # fração do ciclo em que o ejetor fica ligado

    def consumo_por_ciclo_nl(self, cadencia=CADENCIA) -> float:
        if self.consumo_continuo_nl_min > 0:
            return self.consumo_continuo_nl_min * self.fator_utilizacao / cadencia
        area = math.pi * (self.diametro_mm / 1000) ** 2 / 4
        return 2 * area * (self.curso_mm / 1000) * 1000 * RAZAO_COMPRESSAO   # NL

    def consumo_nl_min(self, cadencia=CADENCIA) -> float:
        return self.consumo_por_ciclo_nl(cadencia) * cadencia

class GrafoPneumatico:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float("inf")] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []
        self.consumidores: Dict[str, ConsumidorPneumatico] = {}

    def adicionar_consumidor(self, c: ConsumidorPneumatico):
        self.consumidores[c.tag] = c

    def adicionar_linha(self, origem, destino, comprimento_m, tag_valvula, diam_int_mm):
        u, v = self.v_to_idx[origem], self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        # Resistência linear de Hagen-Poiseuille: R = 128 mu L / (pi D^4)  [Pa.s/m3]
        R = 128 * MU_AR * comprimento_m / (math.pi * (diam_int_mm / 1000) ** 4)
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino, "Comprimento (m)": comprimento_m,
            "Válvula ISA": tag_valvula, "D int (mm)": diam_int_mm, "R": R,
        })

def criar_rede_envasadora():
    nos = ["FRL-001", "ILHA-A", "ILHA-B",
           "CIL-A", "CIL-B", "CIL-C", "BICO",
           "CIL-D", "CIL-E", "EJT-301", "CIL-F", "CIL-G", "CIL-H"]
    g = GrafoPneumatico(nos)
    for c in [
        ConsumidorPneumatico("CIL-A", "Retentor de copos", "100", 16, 25),
        ConsumidorPneumatico("CIL-B", "Válvula 3 vias", "200", 16, 20),
        ConsumidorPneumatico("CIL-C", "Dosador 150 ml", "200", 32, 100),
        ConsumidorPneumatico("BICO", "Abre/fecha bico", "200", 10, 10),
        ConsumidorPneumatico("CIL-D", "Giro do braço", "300", 20, 40),
        ConsumidorPneumatico("CIL-E", "Vertical do braço", "300", 16, 50),
        ConsumidorPneumatico("EJT-301", "Ejetor de vácuo", "300", consumo_continuo_nl_min=15.0, fator_utilizacao=0.4),
        ConsumidorPneumatico("CIL-F", "Prensa seladora", "400", 40, 50),
        ConsumidorPneumatico("CIL-G", "Elevador vertical", "500", 20, 80),
        ConsumidorPneumatico("CIL-H", "Transferência horiz.", "500", 20, 100),
    ]:
        g.adicionar_consumidor(c)
    #                 origem     destino    L(m)  válvula     Dint(mm)
    g.adicionar_linha("FRL-001", "ILHA-A",  2.0, "LINHA-01", 6.5)
    g.adicionar_linha("FRL-001", "ILHA-B",  4.0, "LINHA-02", 6.5)
    g.adicionar_linha("ILHA-A",  "ILHA-B",  1.5, "LINHA-EQ", 6.5)
    g.adicionar_linha("ILHA-A",  "CIL-A",   0.8, "XV-101",   4.0)
    g.adicionar_linha("ILHA-A",  "CIL-B",   1.0, "XV-201",   4.0)
    g.adicionar_linha("ILHA-A",  "CIL-C",   1.2, "XV-202",   4.0)
    g.adicionar_linha("ILHA-A",  "BICO",    1.0, "XV-203",   2.5)
    g.adicionar_linha("ILHA-B",  "CIL-D",   1.0, "XV-301",   4.0)
    g.adicionar_linha("ILHA-B",  "CIL-E",   1.2, "XV-302",   4.0)
    g.adicionar_linha("ILHA-B",  "EJT-301", 1.2, "VAC-301",  4.0)
    g.adicionar_linha("ILHA-B",  "CIL-F",   0.8, "XV-401",   6.5)
    g.adicionar_linha("ILHA-B",  "CIL-G",   1.0, "XV-501",   4.0)
    g.adicionar_linha("ILHA-B",  "CIL-H",   1.5, "XV-502",   4.0)
    return g

rede = criar_rede_envasadora()
n, m = rede.n, len(rede.arestas_detalhes)
rotulos_e = [f"e{j+1}" for j in range(m)]

print(f"=== REDE PNEUMÁTICA: n = {n} vértices, m = {m} arestas ===")
print(formatar_tabela([{"Aresta": rotulos_e[j], "Origem": a["Origem"], "Destino": a["Destino"],
                        "L (m)": a["Comprimento (m)"], "D int (mm)": a["D int (mm)"],
                        "Válvula": a["Válvula ISA"]} for j, a in enumerate(rede.arestas_detalhes)]))

print(f"\n=== CONSUMO DE AR (p = {P_REDE} bar, cadência = {CADENCIA:.0f} copos/min) ===")
tab = [{"Tag": c.tag, "Setor": c.setor, "Função": c.descricao,
        "Ø x curso (mm)": f"{c.diametro_mm:.0f} x {c.curso_mm:.0f}" if c.diametro_mm else "contínuo",
        "NL/ciclo": f"{c.consumo_por_ciclo_nl():.3f}", "NL/min": f"{c.consumo_nl_min():.2f}"}
       for c in rede.consumidores.values()]
print(formatar_tabela(tab))
print(f"Consumo total: {sum(c.consumo_nl_min() for c in rede.consumidores.values()):.2f} NL/min")

=== REDE PNEUMÁTICA: n = 13 vértices, m = 13 arestas ===
Aresta | Origem  | Destino | L (m) | D int (mm) | Válvula 
-------+---------+---------+-------+------------+---------
e1     | FRL-001 | ILHA-A  | 2.0   | 6.5        | LINHA-01
e2     | FRL-001 | ILHA-B  | 4.0   | 6.5        | LINHA-02
e3     | ILHA-A  | ILHA-B  | 1.5   | 6.5        | LINHA-EQ
e4     | ILHA-A  | CIL-A   | 0.8   | 4.0        | XV-101  
e5     | ILHA-A  | CIL-B   | 1.0   | 4.0        | XV-201  
e6     | ILHA-A  | CIL-C   | 1.2   | 4.0        | XV-202  
e7     | ILHA-A  | BICO    | 1.0   | 2.5        | XV-203  
e8     | ILHA-B  | CIL-D   | 1.0   | 4.0        | XV-301  
e9     | ILHA-B  | CIL-E   | 1.2   | 4.0        | XV-302  
e10    | ILHA-B  | EJT-301 | 1.2   | 4.0        | VAC-301 
e11    | ILHA-B  | CIL-F   | 0.8   | 6.5        | XV-401  
e12    | ILHA-B  | CIL-G   | 1.0   | 4.0        | XV-501  
e13    | ILHA-B  | CIL-H   | 1.5   | 4.0        | XV-502  

=== CONSUMO DE AR (p = 6.0 bar, cadência = 12 copos/min) 

## 3. Matriz de Incidência $B$, Soma Nula e Espaço de Ciclos  *(Melhoria M3)*

$$B[i, j] = \begin{cases} -1, & e_j \text{ sai de } v_i \\ +1, & e_j \text{ entra em } v_i \\ 0, & \text{caso contrário} \end{cases}$$

Além da propriedade da soma nula por coluna, calculamos o **posto** de $B$ (esperado $n-1$ para grafo conexo) e a dimensão do **espaço de ciclos** $m-n+1$. Note que a contagem de $-1$ e $+1$ em cada linha fornece os **graus de saída e entrada** de cada nó.

In [11]:
class CalculadorIncidencia:
    @staticmethod
    def construir_matriz_incidencia(vertices, arestas):
        v_idx = {v: i for i, v in enumerate(vertices)}
        B = [[0] * len(arestas) for _ in range(len(vertices))]
        for j, a in enumerate(arestas):
            B[v_idx[a["Origem"]]][j] = -1
            B[v_idx[a["Destino"]]][j] = 1
        return B

B_mat = CalculadorIncidencia.construir_matriz_incidencia(rede.vertices, rede.arestas_detalhes)
print("=== MATRIZ DE INCIDÊNCIA B (13 x 13) ===")
print(formatar_matriz(B_mat, rede.vertices, rotulos_e))

somas_col = [sum(B_mat[i][j] for i in range(n)) for j in range(m)]
assert all(s == 0 for s in somas_col), "Coluna com soma não nula!"
print("\n✓ Soma por coluna nula verificada para todas as", m, "arestas (conservação de massa).")

posto = posto_matriz(B_mat)
print(f"\nPosto(B) = {posto}  (esperado n - 1 = {n - 1}) -> grafo {'CONEXO' if posto == n - 1 else 'DESCONEXO'}")
print(f"Dimensão do espaço de ciclos = m - n + 1 = {m} - {n} + 1 = {m - n + 1}")

print("\n--- Graus obtidos a partir de B ---")
print(formatar_tabela([{"Nó": v, "Grau saída (-1)": B_mat[i].count(-1), "Grau entrada (+1)": B_mat[i].count(1)}
                       for i, v in enumerate(rede.vertices)]))

=== MATRIZ DE INCIDÊNCIA B (13 x 13) ===
        |  e1 |  e2 |  e3 |  e4 |  e5 |  e6 |  e7 |  e8 |  e9 | e10 | e11 | e12 | e13
--------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----
FRL-001 |  -1 |  -1 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0
ILHA-A  |   1 |   0 |  -1 |  -1 |  -1 |  -1 |  -1 |   0 |   0 |   0 |   0 |   0 |   0
ILHA-B  |   0 |   1 |   1 |   0 |   0 |   0 |   0 |  -1 |  -1 |  -1 |  -1 |  -1 |  -1
CIL-A   |   0 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0
CIL-B   |   0 |   0 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0
CIL-C   |   0 |   0 |   0 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |   0 |   0 |   0
BICO    |   0 |   0 |   0 |   0 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |   0 |   0
CIL-D   |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   1 |   0 |   0 |   0 |   0 |   0
CIL-E   |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   0 |   1 |   0 |   0 |   0 |   0
EJT-301 |   0

## 4. Matrizes de Adjacência $A$, de Custos $W$ e Floyd-Warshall  *(Melhorias M2 e M4)*

A matriz de custos usa o **comprimento da mangueira** como peso ($\infty$ = sem ligação direta). O algoritmo de **Floyd-Warshall** calcula, em $O(n^3)$, a menor distância entre **todos os pares** de nós:

$$d_{ij}^{(k)} = \min\left(d_{ij}^{(k-1)},\; d_{ik}^{(k-1)} + d_{kj}^{(k-1)}\right)$$

In [12]:
print("=== MATRIZ DE ADJACÊNCIA BINÁRIA A ===")
print(formatar_matriz(rede.adj_binaria, rede.vertices, rede.vertices))

print("\n=== MATRIZ DE CUSTOS W (comprimento em m) — linhas com saída ===")
linhas_com_saida = [i for i in range(n) if any(rede.adj_binaria[i])]
print(formatar_matriz([rede.adj_pesos[i] for i in linhas_com_saida],
                      [rede.vertices[i] for i in linhas_com_saida], rede.vertices, casas=1))
print("(linhas dos consumidores omitidas: só contêm 0 na diagonal e ∞ — são sumidouros)")

def floyd_warshall(W):
    N = len(W)
    D = [linha[:] for linha in W]
    prox = [[j if W[i][j] != float("inf") else None for j in range(N)] for i in range(N)]
    for k in range(N):
        for i in range(N):
            for j in range(N):
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
                    prox[i][j] = prox[i][k]
    return D, prox

def reconstruir_rota(prox, i, j, nomes):
    if prox[i][j] is None:
        return "sem rota"
    rota = [i]
    while i != j:
        i = prox[i][j]
        rota.append(i)
    return " -> ".join(nomes[k] for k in rota)

D_fw, prox_fw = floyd_warshall(rede.adj_pesos)
origem = rede.v_to_idx["FRL-001"]
print("\n--- Menor comprimento de mangueira a partir do FRL-001 (Floyd-Warshall) ---")
print(formatar_tabela([{"Destino": rede.vertices[j], "Distância (m)": f"{D_fw[origem][j]:.1f}",
                        "Rota": reconstruir_rota(prox_fw, origem, j, rede.vertices)}
                       for j in range(n) if j != origem]))

=== MATRIZ DE ADJACÊNCIA BINÁRIA A ===
        | FRL-001 | ILHA-A | ILHA-B | CIL-A | CIL-B | CIL-C | BICO | CIL-D | CIL-E | EJT-301 | CIL-F | CIL-G | CIL-H
--------+---------+--------+--------+-------+-------+-------+------+-------+-------+---------+-------+-------+------
FRL-001 |       0 |      1 |      1 |     0 |     0 |     0 |    0 |     0 |     0 |       0 |     0 |     0 |     0
ILHA-A  |       0 |      0 |      1 |     1 |     1 |     1 |    1 |     0 |     0 |       0 |     0 |     0 |     0
ILHA-B  |       0 |      0 |      0 |     0 |     0 |     0 |    0 |     1 |     1 |       1 |     1 |     1 |     1
CIL-A   |       0 |      0 |      0 |     0 |     0 |     0 |    0 |     0 |     0 |       0 |     0 |     0 |     0
CIL-B   |       0 |      0 |      0 |     0 |     0 |     0 |    0 |     0 |     0 |       0 |     0 |     0 |     0
CIL-C   |       0 |      0 |      0 |     0 |     0 |     0 |    0 |     0 |     0 |       0 |     0 |     0 |     0
BICO    |       0 |      

## 5. Balanço de Massa de Ar $B\vec{Q} = \vec{S}$ com a 2ª Lei de Kirchhoff  *(Melhoria M5)*

* $\vec{S}$: positivo nos consumidores (sumidouros), negativo no `FRL-001` (fonte), nulo nas ilhas.
* Como existe **1 ciclo independente** (o anel `FRL-001 → ILHA-A → ILHA-B`), as $n-1 = 12$ equações de nó independentes **não bastam** para as $m = 13$ incógnitas: o sistema é **subdeterminado**.
* A equação que falta vem da **Lei das Malhas**: em cada ciclo fundamental $\vec{c}$, a soma das quedas de pressão é nula, $\sum_j c_j R_j Q_j = 0$, com $R_j$ de Hagen-Poiseuille (modelo linear didático).

Sistema completo ($m \times m$): $\begin{bmatrix} B_{red} \\ C \cdot \mathrm{diag}(R) \end{bmatrix} \vec{Q} = \begin{bmatrix} \vec{S}_{red} \\ \vec{0} \end{bmatrix}$

In [13]:
def ciclos_fundamentais(vertices, arestas):
    """Árvore geradora por BFS; cada aresta fora da árvore gera um ciclo fundamental com sinais."""
    viz = {v: [] for v in vertices}
    for j, a in enumerate(arestas):
        viz[a["Origem"]].append((a["Destino"], j))
        viz[a["Destino"]].append((a["Origem"], j))
    pai, fila, arvore = {vertices[0]: (None, None)}, [vertices[0]], set()
    while fila:
        u = fila.pop(0)
        for v, j in viz[u]:
            if v not in pai:
                pai[v] = (u, j); arvore.add(j); fila.append(v)

    def ate_raiz(v):
        cam = []
        while pai[v][0] is not None:
            u, j = pai[v]; cam.append((j, v, u)); v = u
        return cam

    ciclos = []
    for j, a in enumerate(arestas):
        if j in arvore:
            continue
        c = [0] * len(arestas)
        c[j] = 1                                         # percorre u -> v
        pv, pu = ate_raiz(a["Destino"]), ate_raiz(a["Origem"])
        while pv and pu and pv[-1][0] == pu[-1][0]:      # remove trecho comum até o ancestral
            pv.pop(); pu.pop()
        for e, de, para in pv:                           # v sobe até o ancestral
            c[e] += 1 if arestas[e]["Origem"] == de else -1
        for e, de, para in pu:                           # ancestral desce até u
            c[e] += 1 if arestas[e]["Origem"] == para else -1
        ciclos.append(c)
    return ciclos

# Vetor S (NL/min)
S_vec = [0.0] * n
for tag, c in rede.consumidores.items():
    S_vec[rede.v_to_idx[tag]] = c.consumo_nl_min()
S_vec[rede.v_to_idx["FRL-001"]] = -sum(S_vec)

ciclos = ciclos_fundamentais(rede.vertices, rede.arestas_detalhes)
for c in ciclos:
    assert all(v == 0 for v in mult_mat_vet(B_mat, c)), "ciclo fora do núcleo de B"
    print("Ciclo fundamental:", " ".join(f"{'+' if c[j] > 0 else '-'}e{j+1}" for j in range(m) if c[j]),
          "  (B·c = 0 ✓)")

def resolver_vazoes(S):
    ref = rede.v_to_idx["FRL-001"]                     # nó de referência (linha redundante)
    M = [B_mat[i][:] for i in range(n) if i != ref]
    b = [S[i] for i in range(n) if i != ref]
    R = [a["R"] for a in rede.arestas_detalhes]
    for c in ciclos:
        linha = [c[j] * R[j] for j in range(m)]
        esc = max(abs(x) for x in linha)              # normaliza p/ estabilidade numérica
        M.append([x / esc for x in linha]); b.append(0.0)
    return resolver_sistema_linear(M, b)

Q_vec = resolver_vazoes(S_vec)

print("\n--- Vazões resolvidas por aresta ---")
tab_q = []
for j, a in enumerate(rede.arestas_detalhes):
    q_real_m3s = Q_vec[j] / RAZAO_COMPRESSAO / 1000 / 60  # NL/min -> m3/s comprimido
    tab_q.append({"Aresta": rotulos_e[j], "Trecho": f"{a['Origem']}->{a['Destino']}",
                  "Q (NL/min)": f"{Q_vec[j]:.2f}", "ΔP linear (Pa)": f"{a['R'] * q_real_m3s:.1f}"})
print(formatar_tabela(tab_q))

S_calc = mult_mat_vet(B_mat, Q_vec)
print("\n--- Verificação do Balanço (S = B · Q) ---")
print(formatar_tabela([{"Nó": v, "S esperado (NL/min)": f"{S_vec[i]:.2f}", "B·Q (NL/min)": f"{S_calc[i]:.2f}",
                        "Status": "OK" if abs(S_calc[i] - S_vec[i]) < 1e-6 else "ERRO"}
                       for i, v in enumerate(rede.vertices)]))

Ciclo fundamental: +e1 -e2 +e3   (B·c = 0 ✓)

--- Vazões resolvidas por aresta ---
Aresta | Trecho          | Q (NL/min) | ΔP linear (Pa)
-------+-----------------+------------+---------------
e1     | FRL-001->ILHA-A | 26.78      | 53.3          
e2     | FRL-001->ILHA-B | 17.81      | 70.9          
e3     | ILHA-A->ILHA-B  | 11.78      | 17.6          
e4     | ILHA-A->CIL-A   | 0.84       | 4.6           
e5     | ILHA-A->CIL-B   | 0.67       | 4.6           
e6     | ILHA-A->CIL-C   | 13.36      | 111.2         
e7     | ILHA-A->BICO    | 0.13       | 5.9           
e8     | ILHA-B->CIL-D   | 2.09       | 14.5          
e9     | ILHA-B->CIL-E   | 1.67       | 13.9          
e10    | ILHA-B->EJT-301 | 6.00       | 49.9          
e11    | ILHA-B->CIL-F   | 10.44      | 8.3           
e12    | ILHA-B->CIL-G   | 4.18       | 29.0          
e13    | ILHA-B->CIL-H   | 5.22       | 54.3          

--- Verificação do Balanço (S = B · Q) ---
Nó      | S esperado (NL/min) | B·Q (NL/min) | S

## 6. Detecção de Vazamento pelo Resíduo do Balanço  *(Melhoria M6)*

Com transmissores de vazão (FIT) nas linhas, o SCADA calcula $\vec{r} = B\vec{Q}_{med} - \vec{S}_{nominal}$. Em regime sem falha $\vec{r} \approx \vec{0}$; um **vazamento em um nó** aparece como resíduo positivo naquele nó (massa "sumindo") e negativo no `FRL-001` (fonte suprindo o excesso).

Simulamos um vazamento de **4,0 NL/min** em um engate rápido da `ILHA-B`.

In [14]:
VAZAMENTO = 4.0
LIMIAR_ALARME = 0.5   # NL/min (tolerância de medição)

S_real = S_vec[:]
S_real[rede.v_to_idx["ILHA-B"]] += VAZAMENTO
S_real[rede.v_to_idx["FRL-001"]] -= VAZAMENTO
Q_medido = resolver_vazoes(S_real)          # o que os FITs leriam na planta com o vazamento

residuo = [bq - s for bq, s in zip(mult_mat_vet(B_mat, Q_medido), S_vec)]
print(formatar_tabela([{"Nó": v, "Resíduo r (NL/min)": f"{residuo[i]:+.2f}",
                        "Diagnóstico": ("ALARME: vazamento/consumo extra" if residuo[i] > LIMIAR_ALARME else
                                        "Fonte compensando" if residuo[i] < -LIMIAR_ALARME else "OK")}
                       for i, v in enumerate(rede.vertices)]))
nos_alarme = [rede.vertices[i] for i in range(n) if residuo[i] > LIMIAR_ALARME]
print(f"\nNó(s) com suspeita de vazamento: {nos_alarme}")
print(f"Perda anual estimada: {VAZAMENTO * 60 * 16 * 250 / 1000:.0f} Nm³ (16 h/dia, 250 dias/ano)")

Nó      | Resíduo r (NL/min) | Diagnóstico                    
--------+--------------------+--------------------------------
FRL-001 | -4.00              | Fonte compensando              
ILHA-A  | +0.00              | OK                             
ILHA-B  | +4.00              | ALARME: vazamento/consumo extra
CIL-A   | +0.00              | OK                             
CIL-B   | +0.00              | OK                             
CIL-C   | +0.00              | OK                             
BICO    | +0.00              | OK                             
CIL-D   | +0.00              | OK                             
CIL-E   | +0.00              | OK                             
EJT-301 | +0.00              | OK                             
CIL-F   | +0.00              | OK                             
CIL-G   | +0.00              | OK                             
CIL-H   | +0.00              | OK                             

Nó(s) com suspeita de vazamento: ['ILHA-B']
Perda anua

## 7. Laplaciano: $L = BB^T = D - A_{und}$  *(Melhoria M7)*

$D$ é a matriz diagonal de graus e $A_{und} = A + A^T$ a adjacência do grafo não-dirigido subjacente.

In [15]:
L_inc = mult_mat(B_mat, transposta(B_mat))
A_und = [[rede.adj_binaria[i][j] + rede.adj_binaria[j][i] for j in range(n)] for i in range(n)]
L_grau = [[(sum(A_und[i]) if i == j else 0) - A_und[i][j] for j in range(n)] for i in range(n)]
assert L_inc == L_grau
print("✓ B·Bᵀ == D - A  verificado elemento a elemento.\n")
print("=== LAPLACIANO L (13 x 13) ===")
print(formatar_matriz(L_inc, rede.vertices, [v.replace("CIL-", "") for v in rede.vertices]))
print("\nSoma de cada linha de L:", [sum(l) for l in L_inc], "-> vetor 1 pertence ao núcleo de L")

✓ B·Bᵀ == D - A  verificado elemento a elemento.

=== LAPLACIANO L (13 x 13) ===
        | FRL-001 | ILHA-A | ILHA-B |   A |   B |   C | BICO |   D |   E | EJT-301 |   F |   G |   H
--------+---------+--------+--------+-----+-----+-----+------+-----+-----+---------+-----+-----+----
FRL-001 |       2 |     -1 |     -1 |   0 |   0 |   0 |    0 |   0 |   0 |       0 |   0 |   0 |   0
ILHA-A  |      -1 |      6 |     -1 |  -1 |  -1 |  -1 |   -1 |   0 |   0 |       0 |   0 |   0 |   0
ILHA-B  |      -1 |     -1 |      8 |   0 |   0 |   0 |    0 |  -1 |  -1 |      -1 |  -1 |  -1 |  -1
CIL-A   |       0 |     -1 |      0 |   1 |   0 |   0 |    0 |   0 |   0 |       0 |   0 |   0 |   0
CIL-B   |       0 |     -1 |      0 |   0 |   1 |   0 |    0 |   0 |   0 |       0 |   0 |   0 |   0
CIL-C   |       0 |     -1 |      0 |   0 |   0 |   1 |    0 |   0 |   0 |       0 |   0 |   0 |   0
BICO    |       0 |     -1 |      0 |   0 |   0 |   0 |    1 |   0 |   0 |       0 |   0 |   0 |   0
CIL-D   | 

## 8. Grafo da Mesa Indexadora: Potências de $A$, Lead Time e Gargalo  *(Melhoria M8)*

A mesa giratória forma um **ciclo dirigido** $C_5$: Dispensa → Envase → Tampa → Selagem → Ejeção → (volta). Propriedades:

* $(A^k)_{ij}$ = número de caminhos de comprimento $k$ de $i$ a $j$ → indica **onde o copo estará após $k$ indexações**. Para $C_5$, $A^5 = I$.
* Como todas as estações trabalham **em paralelo**, o tempo de ciclo (takt) é $T = \max_i t_i + t_{index}$ → a estação mais lenta é o **gargalo**.
* O custo $W_{i,i+1} = t_i + t_{index}$ permite usar Floyd-Warshall para obter o **lead time ideal** de um copo; numa mesa síncrona, porém, cada passo dura o takt, e o lead time real é $(k-1)\,T + t_{P5}$.

> Os tempos por estação são estimativas de projeto e devem ser substituídos pelos medidos na máquina real.

In [16]:
estacoes = ["P1-Dispensa", "P2-Envase", "P3-Tampa", "P4-Selagem", "P5-Ejecao"]
tempos_s = [1.2, 2.5, 2.2, 2.8, 1.6]        # tempo de operação em cada estação (s)
T_INDEX = 0.8                              # tempo de giro da mesa (s)
k_est = len(estacoes)

A_mesa = [[1 if j == (i + 1) % k_est else 0 for j in range(k_est)] for i in range(k_est)]
print("=== ADJACÊNCIA DA MESA (ciclo dirigido C5) ===")
print(formatar_matriz(A_mesa, estacoes, [e[:2] for e in estacoes]))

print("\n--- Posição de um copo que entra em P1 após k indexações (linha P1 de A^k) ---")
Ak, linhas_pos = [[int(i == j) for j in range(k_est)] for i in range(k_est)], []
for k in range(1, k_est + 1):
    Ak = mult_mat(Ak, A_mesa)
    linhas_pos.append({"k": k, "Linha P1 de A^k": str(Ak[0]), "Copo em": estacoes[Ak[0].index(1)]})
print(formatar_tabela(linhas_pos))
print("A^5 == I ?", Ak == [[int(i == j) for j in range(k_est)] for i in range(k_est)])

W_mesa = [[0.0 if i == j else float("inf") for j in range(k_est)] for i in range(k_est)]
for i in range(k_est):
    W_mesa[i][(i + 1) % k_est] = tempos_s[i] + T_INDEX
D_mesa, prox_mesa = floyd_warshall(W_mesa)
print("\n=== MATRIZ DE DISTÂNCIAS (s) ENTRE ESTAÇÕES — Floyd-Warshall ===")
print(formatar_matriz(D_mesa, estacoes, [e[:2] for e in estacoes], casas=1))

i_garg = max(range(k_est), key=lambda i: tempos_s[i])
takt = tempos_s[i_garg] + T_INDEX
cad_max = 60 / takt
lead_ideal = D_mesa[0][k_est - 1] + tempos_s[-1]          # soma dos custos (Floyd-Warshall)
lead_real = (k_est - 1) * takt + tempos_s[-1]              # mesa síncrona: cada passo dura o takt
print(f"\nGargalo: {estacoes[i_garg]} ({tempos_s[i_garg]} s) -> takt = {takt:.1f} s -> cadência máx = {cad_max:.1f} copos/min")
print(f"Lead time ideal (Floyd-Warshall, sem espera): {lead_ideal:.1f} s")
print(f"Lead time real (mesa síncrona, espera o gargalo): {lead_real:.1f} s "
      f"-> {lead_real - lead_ideal:.1f} s de espera induzida pelo gargalo")
consumo_max = sum(c.consumo_nl_min(cad_max) for c in rede.consumidores.values())
print(f"Consumo de ar na cadência máxima: {consumo_max:.2f} NL/min "
      f"(vs {sum(S_vec[i] for i in range(n) if S_vec[i] > 0):.2f} NL/min na cadência nominal)")

=== ADJACÊNCIA DA MESA (ciclo dirigido C5) ===
            |  P1 |  P2 |  P3 |  P4 |  P5
------------+-----+-----+-----+-----+----
P1-Dispensa |   0 |   1 |   0 |   0 |   0
P2-Envase   |   0 |   0 |   1 |   0 |   0
P3-Tampa    |   0 |   0 |   0 |   1 |   0
P4-Selagem  |   0 |   0 |   0 |   0 |   1
P5-Ejecao   |   1 |   0 |   0 |   0 |   0

--- Posição de um copo que entra em P1 após k indexações (linha P1 de A^k) ---
k | Linha P1 de A^k | Copo em    
--+-----------------+------------
1 | [0, 1, 0, 0, 0] | P2-Envase  
2 | [0, 0, 1, 0, 0] | P3-Tampa   
3 | [0, 0, 0, 1, 0] | P4-Selagem 
4 | [0, 0, 0, 0, 1] | P5-Ejecao  
5 | [1, 0, 0, 0, 0] | P1-Dispensa
A^5 == I ? True

=== MATRIZ DE DISTÂNCIAS (s) ENTRE ESTAÇÕES — Floyd-Warshall ===
            |   P1 |   P2 |   P3 |   P4 |   P5
------------+------+------+------+------+-----
P1-Dispensa |  0.0 |  2.0 |  5.3 |  8.3 | 11.9
P2-Envase   | 12.3 |  0.0 |  3.3 |  6.3 |  9.9
P3-Tampa    |  9.0 | 11.0 |  0.0 |  3.0 |  6.6
P4-Selagem  |  6.0 |  8.

## 9. Conclusões

* A **matriz de incidência** traduz a rede pneumática em um sistema linear de conservação de massa; o **posto** $n-1$ confirma a conectividade e a dimensão $m-n+1 = 1$ revela o anel de equalização.
* Sem a **Lei das Malhas**, a vazão no anel seria indeterminada — o notebook original contornava isso fixando $\vec{Q}$ manualmente.
* O **resíduo do balanço** é um detector de vazamentos barato, que pode virar regra da Base de Conhecimento (Aula 08): `SE residuo_ILHA_B > limiar ENTÃO ALARME_VAZAMENTO`.
* No grafo da mesa, a **selagem** é o gargalo; reduzir seu tempo (ex.: pré-aquecimento da tampa) aumenta diretamente a cadência.